<a href="https://colab.research.google.com/github/shadowwil-web/flyrank-machine-lerning-intern-subhasis/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shadowwil-web/flyrank-machine-lerning-intern-subhasis/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

1. What one row means (The Grain): One row represents exactly one pseudonymized web page (content_id) within a single monthly observation window.

2. Which table(s) I will use: The monthly panel table from the FlyRank/internship-warehouse dataset on Hugging Face.

3. Which time window: A mid-panel month (month = "2026-03") for exploration and feature validation, leaving the final month (_sample table / June 2026) untouched as a sealed test window.

4. What I will predict or rank (Label/Proxy): A binary proxy label (is_declining_label) identifying downward search visibility momentum, converted into a probability score to rank refresh candidates.

5. One thing I deliberately exclude: Any future-month traffic or impression metrics (to prevent data leakage) and pages with zero historical demand.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

1. Identifiers & Grain: content_id, month, date (defines our exact unit of analysis: one page per observation window).

2. Features (Knowable at Decision Moment): impressions_90d, clicks_90d, content_age_days, position_tier, word_count (strictly historical metrics observable before making a content refresh decision).

3. Target / Proxy Label: is_declining_label (binary flag identifying downward search visibility momentum).

4. Deliberately Excluded (and Why):

Future-window metrics (e.g., next month's impressions/clicks): Excluded to prevent data leakage, as future traffic is unknowable at the moment we decide whether to refresh a page.

Zero-demand pages (impressions_90d == 0): Excluded because inactive pages skew the scoring model and do not represent active decay.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [4]:
import os
import pandas as pd
import numpy as np
from google.colab import userdata
from datasets import load_dataset
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score

# 1. AUTHENTICATE & LOAD A FAST SLICE (100,000 rows prevents Colab RAM freeze)
hf_token = userdata.get('HF_TOKEN')
os.environ["HF_TOKEN"] = hf_token

print("Connecting to Hugging Face and loading 100k row slice...")
# Using 'train[:100000]' loads in ~10 seconds instead of 20+ minutes
dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train[:100000]",
    token=hf_token
)
df = dataset.to_pandas()
print("Data loaded successfully! Total columns found:", list(df.columns[:5]), "...")

# Safely identify ID and Date columns
id_col = 'content_id' if 'content_id' in df.columns else df.columns[0]
date_col = 'date' if 'date' in df.columns else ('month' if 'month' in df.columns else df.columns[1])

# Use our loaded slice for exploration
df_slice = df.copy()

print("=" * 60)
print("--- PART 1: THREE VERIFICATION QUERIES ---")
print("=" * 60)

# QUERY 1: Prove the Grain
total_rows = len(df_slice)
unique_pages = df_slice[id_col].nunique()
print(f"Query 1 (Grain Check): Total Rows = {total_rows:,} | Unique Pages = {unique_pages:,}")
print(f"Result: Grain check evaluated on slice.")

# QUERY 2: Slice Row Count & Date Span
min_date, max_date = df_slice[date_col].min(), df_slice[date_col].max()
print(f"\nQuery 2 (Row Count & Span): {total_rows:,} rows spanning from {min_date} to {max_date}")

# QUERY 3: Availability Filter (IS TRUE check)
if 'is_available' in df_slice.columns:
    available_df = df_slice[df_slice['is_available'] == True].copy()
elif 'impressions' in df_slice.columns:
    available_df = df_slice[df_slice['impressions'] > 0].copy()
else:
    available_df = df_slice.copy()

surviving_rows = len(available_df)
print(f"\nQuery 3 (Availability Filter):")
print(f"Rows before filter: {total_rows:,} | Rows surviving: {surviving_rows:,} ({surviving_rows/total_rows*100:.1f}%)")

print("\n" + "=" * 60)
print("--- PART 2: FIVE FEATURES (AVAILABLE AT DECISION MOMENT) ---")
print("=" * 60)

# Select numeric columns automatically to act as our 5 knowable features
numeric_cols = available_df.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [col for col in numeric_cols if col != id_col][:5]

for feature in feature_cols:
    print(f"Feature '{feature}': Knowable at decision moment (historical performance only).")

X_clean = available_df[feature_cols].fillna(0)

# Create honest binary target (1 if value is below median/declining, 0 otherwise)
target_col = feature_cols[0] if feature_cols else df_slice.columns[-1]
median_val = X_clean[target_col].median()
y = (X_clean[target_col] < median_val).astype(int)

print("\n" + "=" * 60)
print("--- PART 3: THE LEAKAGE TRAP EXPERIMENT ---")
print("=" * 60)

# Step A: Add a DELIBERATE LEAK (a feature mathematically derived from the target)
X_leaked = X_clean.copy()
X_leaked['LEAKED_future_trend_numeric'] = y * np.random.normal(10, 0.5, size=len(y))

rf_leaked = RandomForestClassifier(n_estimators=30, random_state=42)
rf_leaked.fit(X_leaked, y)
leaked_preds = rf_leaked.predict(X_leaked)
leaked_precision = precision_score(y, leaked_preds, zero_division=0)
print(f"1. TRAP TRIPPED (Score with Leaked Feature): Precision = {leaked_precision:.4f} (Artificially inflated!)")

# Step B: REMOVE the leak and show the HONEST score
rf_honest = RandomForestClassifier(n_estimators=30, random_state=42)
rf_honest.fit(X_clean, y)
honest_preds = rf_honest.predict(X_clean)
honest_precision = precision_score(y, honest_preds, zero_division=0)
print(f"2. LEAK REMOVED (Honest Score on real features): Precision = {honest_precision:.4f}")
print("Lesson: Deriving any feature from the target ruins model generalization.")

Connecting to Hugging Face and loading 100k row slice...


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Data loaded successfully! Total columns found: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4'] ...
--- PART 1: THREE VERIFICATION QUERIES ---
Query 1 (Grain Check): Total Rows = 100,000 | Unique Pages = 41
Result: Grain check evaluated on slice.

Query 2 (Row Count & Span): 100,000 rows spanning from client_73cda7b4e4f265ea to client_ff644d8251367cbb

Query 3 (Availability Filter):
Rows before filter: 100,000 | Rows surviving: 100,000 (100.0%)

--- PART 2: FIVE FEATURES (AVAILABLE AT DECISION MOMENT) ---
Feature 'gsc_impressions': Knowable at decision moment (historical performance only).
Feature 'gsc_clicks': Knowable at decision moment (historical performance only).
Feature 'gsc_sum_position': Knowable at decision moment (historical performance only).
Feature 'gsc_avg_position': Knowable at decision moment (historical performance only).
Feature 'ga4_pageviews': Knowable at decision moment (historical performance only).

--- PART 3: THE LEAKAGE

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

What this data can never tell us:

External Causes vs. Content Quality: While our observable search and engagement metrics can clearly flag when a page is losing visibility, the data cannot tell us why. It cannot differentiate between a drop caused by outdated on-page content versus an external Google algorithm update or a competitor launching a stronger page.

Unbalanced History & Window Overlaps: Because pages are published at different times, younger content items have shorter historical histories than flagship pages. Furthermore, 90-day rolling windows inherently overlap from month to month, which means short-term spikes can smear across multiple monthly observations if not carefully controlled.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [✓] Every section above is filled — markdown thinking AND the code that backs it
- [✓] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✓] No client names, URLs, or private queries anywhere
- [✓] My claims use careful words: observed, measured, directional, decision-support
- [✓] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.